# Part A pilot: live A0–A4 context ablation (Gemini free tier)

Runs `python3 -m ablation` against a reviewed items file on Google Drive. Run the cells in order. The live cells make real API calls, so each one stops at a hard `--max-calls` cap.

**Before running:**
- Items are built locally with `draft-items` and `build-items`, then uploaded to `MyDrive/ablation-pilot/items/` with their images.
- `GEMINI_API_KEY` is in Colab Secrets (never printed).
- Confirm `MODEL` is free-tier with image input, and set `MIN_INTERVAL` from your AI Studio limits. The pilot sends only public forum quotes and your own images, and free-tier content may be used to improve Google's products.


In [ ]:
from google.colab import drive, userdata
import datetime, json, os, pathlib

drive.mount('/content/drive')

REPO_COMMIT = "REPLACE_WITH_REVIEWED_COMMIT"  # pin the commit you reviewed
MODEL = "gemini-3.8-flash"                    # confirm free-tier + image input in AI Studio first
MIN_INTERVAL = 12.0                           # seconds between calls; set from your RPM limit
MAX_OUTPUT_TOKENS = 2048                      # thinking may count against this limit
DRIVE = pathlib.Path('/content/drive/MyDrive/ablation-pilot')
ITEMS = DRIVE / 'items' / 'items.jsonl'       # built by build-items; images sit beside it
os.environ['GEMINI_API_KEY'] = userdata.get('GEMINI_API_KEY')  # never printed
STAMP = datetime.datetime.now(datetime.timezone.utc).strftime('%Y%m%d-%H%M%S')


In [ ]:
!git clone -q https://github.com/BobbY-24/llm-eval-ablation-multimodal.git /content/repo
%cd /content/repo
!git checkout -q {REPO_COMMIT}
!python3 -m ablation plan --items {ITEMS} --task mcq


## Smoke run: 2 items × 5 conditions, at most 12 calls

In [ ]:
SMOKE = DRIVE / 'items' / 'smoke.jsonl'
SMOKE.write_text(''.join(ITEMS.read_text().splitlines(keepends=True)[:2]), encoding='utf-8')
SMOKE_OUT = DRIVE / 'runs' / f'smoke-{STAMP}'
!python3 -m ablation run --provider gemini --model {MODEL} --live --items {SMOKE} --task mcq --max-calls 12 --transient-retries 2 --min-interval {MIN_INTERVAL} --max-output-tokens {MAX_OUTPUT_TOKENS} --out {SMOKE_OUT}


In [ ]:
# Status and usage per call only; raw model text is not printed.
for line in (SMOKE_OUT / 'attempts.jsonl').read_text(encoding='utf-8').splitlines():
    r = json.loads(line)
    print(r['example_id'], r['condition'], 'repair' if r['attempt'] else 'first',
          'retry' + str(r['transient_try']), r['parsing_status'], r['error'] or '',
          (r['usage'] or {}).get('totalTokenCount'))


Continue only if every smoke call is `valid` (or the error is understood) and no response was cut off by the output limit.

## Full run: all reviewed items, at most 260 calls

In [ ]:
RUN_DIR = DRIVE / 'runs' / f'pilot-{MODEL}-{STAMP}'
!python3 -m ablation run --provider gemini --model {MODEL} --live --items {ITEMS} --task mcq --max-calls 260 --transient-retries 2 --min-interval {MIN_INTERVAL} --max-output-tokens {MAX_OUTPUT_TOKENS} --out {RUN_DIR}
!python3 -m ablation score --items {ITEMS} --predictions {RUN_DIR}/predictions.jsonl --out {RUN_DIR}/score.json


## Descriptive breakdown by stratum (not a confirmatory test)

In [ ]:
import pandas as pd
from ablation.conditions import load_items
from ablation.scoring import load_predictions, score_rows

items = load_items(ITEMS)
scored = pd.DataFrame(score_rows(load_predictions(RUN_DIR / 'predictions.jsonl'), items))
meta = pd.DataFrame([{'example_id': i.id, 'text_explicitness': i.metadata['text_explicitness'],
                      'visual_groundability': i.metadata['visual_groundability']} for i in items])
df = scored.merge(meta, on='example_id')
for key in ('text_explicitness', 'visual_groundability', 'community'):
    display(df.groupby(['condition', key])['score'].agg(['mean', 'count']).round(2))
